Chargement sécurisé

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

if connection_string is None:
    raise ValueError("Connection string non trouvée dans .env")

print("Connexion sécurisée OK")

Connexion sécurisée OK


Connexion Blob

In [2]:
from azure.storage.blob import BlobServiceClient

container_name = "documents"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)

print("Connexion Blob OK")

Connexion Blob OK


---

Upload récursif

In [3]:
import os

def upload_folder(folder_path):
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            local_path = os.path.join(root, file)
            
            blob_name = os.path.relpath(local_path, folder_path).replace("\\", "/")
            
            print(f"Upload en cours : {local_path}")  # DEBUG
            
            with open(local_path, "rb") as data:
                container_client.upload_blob(name=blob_name, data=data, overwrite=True)
            
            print(f"Upload OK : {blob_name}")

upload_folder("data")

Upload en cours : data\AMM_Hydraulic_Pump.pdf
Upload OK : AMM_Hydraulic_Pump.pdf
Upload en cours : data\Capture d'écran 2026-03-26 123338.png
Upload OK : Capture d'écran 2026-03-26 123338.png
Upload en cours : data\Maintenance_Report.pdf
Upload OK : Maintenance_Report.pdf


---

Extraction du texte PDF

In [4]:
from pypdf import PdfReader

def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    
    for page in reader.pages:
        text += page.extract_text() + "\n"
    
    return text

# Test
text = extract_text_from_pdf("data/AMM_Hydraulic_Pump.pdf")
print(text[:1000])

AMM - Hydraulic Pump Maintenance
1. General Description
This section describes the hydraulic pump system installed on the aircraft. The system provides
pressure to flight control actuators and landing gear.
2. Removal Procedure
Depressurize the hydraulic system. Disconnect electrical connectors. Remove mounting bolts.
Carefully extract the pump from the housing.
3. Installation Procedure
Position the pump. Install mounting bolts with torque specification. Reconnect electrical connectors.
Perform leak test and operational check.
4. Safety Precautions
Ensure aircraft is powered down. Wear protective equipment. Follow lockout-tagout procedures.




---

Chunking intelligent  

_On va découper par sections métier_

In [6]:
import re

def smart_chunking(text):
    # patterns de sections (adapté aviation)
    section_patterns = [
        r"\n\d+\.\s+[A-Z][^\n]+",  # 1. Title
    ]
    
    # split sur les sections
    sections = re.split(r"\n(?=\d+\.\s+)", text)
    
    chunks = []
    
    for section in sections:
        if len(section.strip()) < 50:
            continue
            
        chunks.append({
            "content": section.strip(),
            "length": len(section),
        })
    
    return chunks

# Test
chunks = smart_chunking(text)

for i, chunk in enumerate(chunks):
    print(f"\n--- CHUNK {i} ---\n")
    print(chunk["content"][:300])


--- CHUNK 0 ---

1. General Description
This section describes the hydraulic pump system installed on the aircraft. The system provides
pressure to flight control actuators and landing gear.

--- CHUNK 1 ---

2. Removal Procedure
Depressurize the hydraulic system. Disconnect electrical connectors. Remove mounting bolts.
Carefully extract the pump from the housing.

--- CHUNK 2 ---

3. Installation Procedure
Position the pump. Install mounting bolts with torque specification. Reconnect electrical connectors.
Perform leak test and operational check.

--- CHUNK 3 ---

4. Safety Precautions
Ensure aircraft is powered down. Wear protective equipment. Follow lockout-tagout procedures.


---

Enrichir avec métadonnées 

_Permettent de faire:_     
_- filtres_          
_- analyse avancée_             
_- réponses intelligentes_          

In [7]:
import os

def add_metadata(chunks, source_file):
    enriched_chunks = []
    
    # Titre du document = nom du fichier sans extension
    title = os.path.splitext(os.path.basename(source_file))[0]
    
    for chunk in chunks:
        content = chunk["content"]
        
        enriched_chunks.append({
            "content": content,
            "source": source_file,
            "title": title,  # <-- nouveau champ
            "type": "AMM" if "AMM" in source_file else "REPORT",
            "section": content.split("\n")[0],  # titre de section
            "length": len(content)
        })
    
    return enriched_chunks

Test

In [8]:
enriched_chunks = add_metadata(chunks, "AMM_Hydraulic_Pump.pdf")

for c in enriched_chunks:
    print("\n---")
    print("Title:", c["title"])
    print("Section:", c["section"])
    print("Type:", c["type"])
    print("Source:", c["source"])
    print("Length:", c["length"])



---
Title: AMM_Hydraulic_Pump
Section: 1. General Description
Type: AMM
Source: AMM_Hydraulic_Pump.pdf
Length: 173

---
Title: AMM_Hydraulic_Pump
Section: 2. Removal Procedure
Type: AMM
Source: AMM_Hydraulic_Pump.pdf
Length: 157

---
Title: AMM_Hydraulic_Pump
Section: 3. Installation Procedure
Type: AMM
Source: AMM_Hydraulic_Pump.pdf
Length: 168

---
Title: AMM_Hydraulic_Pump
Section: 4. Safety Precautions
Type: AMM
Source: AMM_Hydraulic_Pump.pdf
Length: 115


---

Embeddings + Azure AI Search

_Connexion OpenAI (embeddings)_

In [9]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-02-01",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")

_Générarion d'un embedding_

In [10]:
def get_embedding(text):
    response = client.embeddings.create(
        model=embedding_model,
        input=text
    )
    return response.data[0].embedding

# Test
emb = get_embedding("Hydraulic pump removal procedure")
print(len(emb))

3072


_Création l’index Azure AI Search_

In [11]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration
)
from azure.core.credentials import AzureKeyCredential
import os

search_client = SearchIndexClient(
    endpoint=os.getenv("AZURE_SEARCH_ENDPOINT"),
    credential=AzureKeyCredential(os.getenv("AZURE_SEARCH_KEY"))
)

index_name = os.getenv("AZURE_SEARCH_INDEX_NAME")

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchField(name="title", type=SearchFieldDataType.String),
    SearchField(name="content", type=SearchFieldDataType.String),
    SearchField(name="source", type=SearchFieldDataType.String),
    SearchField(name="type", type=SearchFieldDataType.String),
    SearchField(name="section", type=SearchFieldDataType.String),
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        vector_search_dimensions=3072,  # car text-embedding-3-large
        vector_search_profile_name="my-vector-profile"
    )
]

vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(name="my-algorithm")
    ],
    profiles=[
        VectorSearchProfile(
            name="my-vector-profile",
            algorithm_configuration_name="my-algorithm"
        )
    ]
)

index = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search
)

search_client.create_index(index)

print("Index créé")

Index créé


_Indexation des chunks_

In [12]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
import hashlib
import os

search_client = SearchClient(
    endpoint=os.getenv("AZURE_SEARCH_ENDPOINT"),
    index_name=os.getenv("AZURE_SEARCH_INDEX_NAME"),
    credential=AzureKeyCredential(os.getenv("AZURE_SEARCH_KEY"))
)

def make_id(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def index_chunks(chunks):
    documents = []
    
    for chunk in chunks:
        embedding = get_embedding(chunk["content"])
        
        documents.append({
            "id": make_id(chunk["content"]),   # ID stable
            "title": chunk["title"],
            "content": chunk["content"],
            "source": chunk["source"],
            "type": chunk["type"],
            "section": chunk["section"],
            "embedding": embedding
        })
    
    result = search_client.upload_documents(documents)
    print(result)
    print(f"{len(documents)} documents indexés (mise à jour ou ajout)")

_Lancement_

In [13]:
index_chunks(enriched_chunks)

[<azure.search.documents._generated.models._models_py3.IndexingResult object at 0x0000013CB5049D10>, <azure.search.documents._generated.models._models_py3.IndexingResult object at 0x0000013CB53C42D0>, <azure.search.documents._generated.models._models_py3.IndexingResult object at 0x0000013CB53C4090>, <azure.search.documents._generated.models._models_py3.IndexingResult object at 0x0000013CB53C4690>]
4 documents indexés (mise à jour ou ajout)


---

Requêtage RAG

_Fonction de recherche vectorielle_

In [13]:
from azure.search.documents.models import VectorizedQuery

def search_documents(query, doc_type=None):

    query_vector = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=3,
        fields="embedding"
    )

    # 🔥 filtre OCR / AMM / REPORT
    filter_query = None
    if doc_type:
        filter_query = f"type eq '{doc_type}'"

    results = search_client.search(
        search_text="",
        vector_queries=[vector_query],
        filter=filter_query
    )

    docs = []

    for result in results:
        docs.append({
            "content": result["content"],
            "source": result.get("source", ""),
            "type": result.get("type", ""),
            "section": result.get("section", "")
        })

    return docs

_Génération avec text-embedding-3-large_


In [14]:
chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

def generate_answer(query, context_docs):

    # 🔥 extraire uniquement le texte
    context = "\n\n".join([doc["content"] for doc in context_docs])
    
    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": "You are an aircraft maintenance expert. Answer ONLY using the provided context."
            },
            {
                "role": "user",
                "content": f"""
Context:
{context}

Question:
{query}
"""
            }
        ]
    )

    return response.choices[0].message.content

_Pipeline RAG complet_

In [15]:
def rag_query(query, doc_type=None):

    docs = search_documents(query, doc_type=doc_type)

    answer = generate_answer(query, docs)

    return answer, docs

Test

In [16]:
answer, docs = rag_query("What is the procedure to remove the hydraulic pump?")
print(answer)

The removal procedure for the hydraulic pump is as follows:
1. Depressurize the hydraulic system.
2. Disconnect electrical connectors.
3. Remove mounting bolts.
4. Carefully extract the pump from the housing.


In [17]:
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
import os

# ==============================
# 🔹 CLIENT VISION
# ==============================
vision_client = ImageAnalysisClient(
    endpoint=os.getenv("AZURE_VISION_ENDPOINT"),
    credential=AzureKeyCredential(os.getenv("AZURE_VISION_KEY"))
)

# ==============================
# 🔹 TEST OCR
# ==============================
def test_ocr(image_path):

    with open(image_path, "rb") as f:
        image_data = f.read()

    result = vision_client.analyze(
        image_data=image_data,
        visual_features=[VisualFeatures.READ]
    )

    text = ""

    if result.read:
        for block in result.read.blocks:
            for line in block.lines:
                text += line.text + "\n"

    return text

In [18]:
text = test_ocr(r"C:\Users\ghood\Desktop\PF_AML\RAG\data\Capture d'écran 2026-03-26 123338.png")
print(text)

MACHINE LEARNING
Intelligences Artificielles (IA) / Artificial Intelligences (AI)
Apprentissage Supervisé - Python
SUPERVESCO
UNSUPERVISED
REINFORCEMENT
Exercice pratique
Implémentez en Python la suite de Fibonacci (0, 1, 1, 2, 3, 5, 8, 13, 21, ... ) qui
· part de 2 nombres a=0 et b=1,
. et qui calcule le nombre suivant en additionnant les 2 nombres précédents.
Indices :
· Vous "imprimerez" cette suite jusqu'a atteindre un nombre n que vous définirez
· dans Python il est possible de mettre à jour 2 variables simultanément sur la même lignes : a, b = b, a+b



Nettoyage des caractères du OCR

In [19]:
def build_chunks_from_ocr(text):

    # 🔹 Nettoyage simple
    text = text.replace("\r", "\n")
    text = text.strip()

    # 🔹 Split intelligent
    raw_chunks = text.split("\n")

    chunks = []
    buffer = ""

    for line in raw_chunks:

        line = line.strip()

        if not line:
            continue

        buffer += " " + line

        # 🔥 règle de chunk (~400 caractères)
        if len(buffer) > 400:
            chunks.append(buffer.strip())
            buffer = ""

    if buffer:
        chunks.append(buffer.strip())

    return chunks

In [20]:
text = test_ocr(r"C:\Users\ghood\Desktop\PF_AML\RAG\data\Capture d'écran 2026-03-26 123338.png")

chunks = build_chunks_from_ocr(text)

print(len(chunks))
print(chunks[0])

2
MACHINE LEARNING Intelligences Artificielles (IA) / Artificial Intelligences (AI) Apprentissage Supervisé - Python SUPERVESCO UNSUPERVISED REINFORCEMENT Exercice pratique Implémentez en Python la suite de Fibonacci (0, 1, 1, 2, 3, 5, 8, 13, 21, ... ) qui · part de 2 nombres a=0 et b=1, . et qui calcule le nombre suivant en additionnant les 2 nombres précédents. Indices : · Vous "imprimerez" cette suite jusqu'a atteindre un nombre n que vous définirez


Indexation des chunks OCR

In [21]:
def ingest_ocr_pipeline(image_path):

    # 1. OCR
    text = test_ocr(image_path)

    # 2. Chunking (ta fonction étape 3)
    raw_chunks = build_chunks_from_ocr(text)

    # 3. Structuration (compatible avec TON index_chunks)
    chunks = []

    for i, chunk in enumerate(raw_chunks):

        chunks.append({
            "content": chunk,
            "source": "OCR_UPLOAD",
            "type": "OCR",
            "section": f"OCR_{i}"
        })

    # 4. Indexation (TON code existant)
    index_chunks(chunks)

    return len(chunks)

In [22]:
nb = ingest_ocr_pipeline(r"C:\Users\ghood\Desktop\PF_AML\RAG\data\Capture d'écran 2026-03-26 123338.png")
print(f"{nb} chunks indexés")

[<azure.search.documents._generated.models._models_py3.IndexingResult object at 0x00000265E40BC350>, <azure.search.documents._generated.models._models_py3.IndexingResult object at 0x00000265E40DF190>]
2 documents indexés (mise à jour ou ajout)
2 chunks indexés


In [23]:
answer, docs = rag_query(
    "Que décrit ce document",
    doc_type="OCR"
)

print(answer)

Ce document décrit un exercice pratique en Python consistant à implémenter la suite de Fibonacci. Il précise que la suite commence par deux nombres (a=0 et b=1) et que chaque nombre suivant est calculé par la somme des deux précédents. Il donne un indice sur la manière de mettre à jour simultanément deux variables en Python (a, b = b, a+b) et indique que la suite doit être imprimée jusqu'à atteindre un nombre n défini par l'utilisateur.
